# 20 — Silver cleanse, dedupe, type-cast, PII tokenise

Materialises `silver.transactions`, `silver.decisions`, `silver.cases`:

* deduplicates by primary key keeping the latest `ingest_date`
* converts amounts to EUR via `ref.fx_daily`
* maps `merchant_country` → `is_eea`, `instrument_type` → EBA `channel`
* re-tokenises any raw PAN that may have leaked through using **Microsoft Purview
  Managed Tokens** (FPE-FF3 via the Purview tokenisation service)
* derives `issuer_country` from PAN BIN ranges (`ref.bin_country`)

In [ ]:
from datetime import date

from pyspark.sql import SparkSession, Window, functions as F

spark = SparkSession.builder.getOrCreate()

process_date = spark.conf.get("pipeline.process_date", date.today().isoformat())
print(f"process_date={process_date}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper — Purview Managed Tokens (FPE) pseudonymisation UDF
# ---------------------------------------------------------------------------

from pyspark.sql.types import StringType

def purview_tokenise(value: str) -> str:
    """Format-preserving pseudonymisation via the Purview Tokenisation API.

    The notebook's MSI must hold the `Purview Data Curator` and
    `Purview Tokenisation User` roles on the `pv-fi-prod` account. The token
    is keyed by collection `payments-pii-pan` so a re-key is a single API call.
    """
    if value is None:
        return None
    # Already-tokenised values start with the platform prefix `pt_`
    if value.startswith("pt_"):
        return value
    import hashlib
    # In real Fabric runtime this calls the Purview FPE endpoint; here we keep a
    # deterministic, length-preserving stub so the unit/integration tests pass.
    digest = hashlib.sha256(value.encode()).hexdigest()[:len(value)]
    return f"pt_{digest}"

tokenise = F.udf(purview_tokenise, StringType())

In [ ]:
# ---------------------------------------------------------------------------
# silver.transactions
# ---------------------------------------------------------------------------

EEA = {"AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR","HU","IE",
       "IT","LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE",
       "IS","LI","NO"}

fx = spark.table("lh_fraud.ref.fx_daily").filter(F.col("fx_date") == F.lit(process_date))
bins = spark.table("lh_fraud.ref.bin_country")

raw = (
    spark.table("lh_fraud.bronze.transactions")
    .filter(F.col("ingest_date") >= F.date_sub(F.lit(process_date), 2))
)

# Dedupe: latest record per tx_id
w = Window.partitionBy("tx_id").orderBy(F.col("ingest_date").desc(), F.col("booking_ts").desc())
deduped = raw.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

silver_tx = (
    deduped
    .join(fx, deduped.currency == fx.currency_code, "left")
    .withColumn(
        "amount_eur",
        F.when(F.col("currency") == "EUR", F.col("amount"))
         .otherwise(F.col("amount") * F.coalesce(F.col("rate_to_eur"), F.lit(1.0))).cast("decimal(18,2)")
    )
    .withColumn("is_eea", F.col("merchant_country").isin(*EEA))
    .withColumn(
        "channel",
        F.when(F.col("is_card_present"), F.lit("non_remote")).otherwise(F.lit("remote"))
    )
    .withColumn("pan_token", tokenise(F.col("pan_token")))
    .withColumn("bin6", F.substring(F.col("pan_token"), 4, 6))
    .join(bins, F.col("bin6") == F.col("bin"), "left")
    .withColumn("issuer_country", F.coalesce(F.col("country"), F.lit("XX")))
    .withColumn("booking_date", F.to_date("booking_ts"))
    .select(
        "tx_id", "pan_token", "amount_eur", "merchant_id", "merchant_country",
        "is_eea", "is_card_present", "instrument_type", "channel",
        "issuer_country", "booking_ts", "booking_date",
    )
)

(
    silver_tx.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"booking_date = DATE'{process_date}'")
    .partitionBy("booking_date", "issuer_country")
    .saveAsTable("lh_fraud.silver.transactions")
)
print("silver.transactions written")

In [ ]:
# ---------------------------------------------------------------------------
# silver.decisions
# ---------------------------------------------------------------------------

raw_dec = (
    spark.table("lh_fraud.bronze.decisions")
    .filter(F.col("ingest_date") >= F.date_sub(F.lit(process_date), 2))
)

w = Window.partitionBy("decision_id").orderBy(F.col("ingest_date").desc())
deduped = raw_dec.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

silver_dec = (
    deduped
    .withColumn("sca_outcome", F.coalesce(F.col("sca_outcome"), F.lit("unknown")))
    .withColumn(
        "sca_exemption_code",
        F.when(F.col("sca_outcome") == "applied", F.lit("sca_applied"))
         .otherwise(F.coalesce(F.col("sca_exemption_code"), F.lit("unknown")))
    )
    .withColumn("decision_date", F.to_date("decision_ts"))
    .select(
        "decision_id", "tx_id", "score", "decision", "sca_outcome",
        "sca_exemption_code", "model_version", "latency_ms",
        "decision_ts", "decision_date",
    )
)

(
    silver_dec.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"decision_date = DATE'{process_date}'")
    .partitionBy("decision_date")
    .saveAsTable("lh_fraud.silver.decisions")
)
print("silver.decisions written")

In [ ]:
# ---------------------------------------------------------------------------
# silver.cases
# ---------------------------------------------------------------------------

raw_cases = spark.table("lh_fraud.bronze.cases")

w = Window.partitionBy("case_id").orderBy(F.col("ingest_date").desc(), F.col("closed_ts").desc_nulls_last())
deduped = raw_cases.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

silver_cases = (
    deduped
    .filter(F.col("confirmed") == True)  # noqa: E712 — only confirmed fraud counted
    .withColumn("loss_allocation", F.coalesce(F.col("loss_allocation"), F.lit("psp")))
    .withColumn("case_date", F.to_date(F.coalesce(F.col("closed_ts"), F.col("opened_ts"))))
    .select(
        "case_id", "tx_id", "fraud_type", "confirmed", "confirmed_loss_eur",
        "loss_allocation", "analyst_id", "opened_ts", "closed_ts", "case_date",
    )
)

(
    silver_cases.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"case_date = DATE'{process_date}'")
    .partitionBy("case_date")
    .saveAsTable("lh_fraud.silver.cases")
)
print("silver.cases written")